In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║        ECOAUDIT AI — FULL NOTEBOOK (Google Colab)           ║
# ║  Universal Biomass & Carbon Inference from Any Coordinate   ║
# ╚══════════════════════════════════════════════════════════════╝
#
# HOW TO USE IN COLAB:
# ─────────────────────────────────────────────────────────────────
#  CELL 1 ── Install dependencies & authenticate GEE  (run once)
#  CELL 2 ── Core inference engine                    (run once)
#  CELL 3 ── Manual coordinate checker                (re-run anytime)
# ─────────────────────────────────────────────────────────────────


# ================================================================
# CELL 1 — SETUP  (run this first, once per session)
# ================================================================

# !pip install earthengine-api global-land-mask scikit-learn pandas numpy --quiet

import ee
import pandas as pd
import numpy as np
import pickle
import math

# Authenticate once (opens browser for Google login), then initialise
ee.Authenticate()   # ← uncomment and run ONCE the very first time
ee.Initialize(project='ecoaudit-ai-498509')

print("✅  Google Earth Engine initialised.")


# ================================================================
# CELL 2 — CORE INFERENCE ENGINE
# ================================================================

# ── MODIS Land Cover classes that mean "no vegetation / no biomass" ──────────
# 0  = Water
# 11 = Permanent Wetlands  (excluded — no AGB model support)
# 13 = Urban and Built-up  (excluded — buildings skew AGB)
# 15 = Snow and Ice        ← KEY FIX: deep Antarctic/Arctic ice treated as 0-biomass
# 17 = Water Bodies
NON_VEGETATED_CLASSES = {0, 11, 13, 15, 17}

# Surface type labels for human-readable output
SURFACE_LABELS = {
    0:  "Ocean/Water",
    11: "Permanent Wetland",
    13: "Urban/Built-up",
    15: "Permanent Ice/Snow",
    17: "Water Body",
}


# ────────────────────────────────────────────────────────────────
# FUNCTION 1 — Surface classification
# ────────────────────────────────────────────────────────────────

def classify_surface(lat: float, lon: float) -> tuple[bool, str]:
    """
    Classifies the surface type at a coordinate using MODIS MCD12Q1
    land cover (500 m resolution) via Google Earth Engine.

    Returns
    -------
    (is_vegetated_land : bool, surface_label : str)

    Surface labels
    ──────────────
    "Ocean/Water"        → open ocean, no MODIS pixel at all
    "Permanent Ice/Snow" → MODIS class 15  (Antarctic/Greenland ice sheets)
    "Urban/Built-up"     → MODIS class 13
    "Permanent Wetland"  → MODIS class 11
    "Water Body"         → MODIS class 17
    "Land"               → all other MODIS classes (vegetated or bare soil)
    """
    point = ee.Geometry.Point([lon, lat])
    try:
        result = (
            ee.ImageCollection("MODIS/006/MCD12Q1")
            .filterDate("2020-01-01", "2021-01-01")
            .first()
            .select("LC_Type1")
            .reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=500,
                maxPixels=1
            )
            .getInfo()
        )
        lc_value = result.get("LC_Type1")

        # No MODIS pixel = open ocean (MODIS only covers land + coastal water)
        if lc_value is None:
            return False, "Ocean/Water"

        lc_int = int(lc_value)

        if lc_int in NON_VEGETATED_CLASSES:
            label = SURFACE_LABELS.get(lc_int, f"Non-vegetated (class {lc_int})")
            return False, label

        return True, "Land"

    except Exception as exc:
        # Last-resort fallback: coarse global_land_mask
        try:
            from global_land_mask import globe
            on_land = bool(globe.is_land(lat, lon))
            return on_land, ("Land" if on_land else "Ocean/Water")
        except Exception:
            return False, f"Classification error: {exc}"


# ────────────────────────────────────────────────────────────────
# FUNCTION 2 — Live GEE feature extraction
# ────────────────────────────────────────────────────────────────

def fetch_gee_features(lat: float, lon: float) -> dict | None:
    """
    Fetches real satellite-derived features for any vegetated land
    coordinate via Google Earth Engine.

    Data sources
    ────────────
    • Sentinel-2 SR  (optical bands B2/B4/B5/B6/B7/B8 + NDVI/EVI/NDRE)
    • Sentinel-1 GRD (SAR backscatter VV, VH)
    • ERA5-Land      (2 m air temperature °C, total precipitation mm/yr)
    • SRTM           (terrain elevation m)

    Returns a feature dict, or None if no usable data is available.
    """
    point  = ee.Geometry.Point([lon, lat])
    region = point.buffer(1000)   # 1 km buffer for stable pixel sampling

    features = {}

    # ── Sentinel-2 optical ────────────────────────────────────────────────────
    try:
        s2 = (
            ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(point)
            .filterDate("2023-01-01", "2024-01-01")
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
            .median()
            .select(["B2", "B4", "B5", "B6", "B7", "B8"])
        )
        optical = s2.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=10,
            maxPixels=1e6
        ).getInfo()

        for band in ["B2", "B4", "B5", "B6", "B7", "B8"]:
            features[band] = optical.get(band, np.nan)

        # Vegetation indices — safe division with epsilon
        b2 = features["B2"] if features["B2"] and not (isinstance(features["B2"], float) and math.isnan(features["B2"])) else 0.0
        b4 = features["B4"] if features["B4"] and not (isinstance(features["B4"], float) and math.isnan(features["B4"])) else 1e-6
        b5 = features["B5"] if features["B5"] and not (isinstance(features["B5"], float) and math.isnan(features["B5"])) else 0.0
        b8 = features["B8"] if features["B8"] and not (isinstance(features["B8"], float) and math.isnan(features["B8"])) else 0.0

        eps = 1e-6
        ndvi = (b8 - b4) / (b8 + b4 + eps)
        evi  = 2.5 * (b8 - b4) / (b8 + 6.0 * b4 - 7.5 * b2 + 10000.0 + eps)
        ndre = (b5 - b4) / (b5 + b4 + eps)

        features["NDVI"] = round(float(ndvi), 6)
        features["EVI"]  = round(float(evi),  6)
        features["NDRE"] = round(float(ndre), 6)

    except Exception as exc:
        print(f"  ⚠️  Sentinel-2 fetch failed: {exc}")
        for k in ["B2", "B4", "B5", "B6", "B7", "B8", "NDVI", "EVI", "NDRE"]:
            features[k] = np.nan

    # ── Sentinel-1 SAR ────────────────────────────────────────────────────────
    try:
        s1 = (
            ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(point)
            .filterDate("2023-01-01", "2024-01-01")
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .select(["VV", "VH"])
            .mean()
        )
        radar = s1.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=10,
            maxPixels=1e6
        ).getInfo()
        features["VV"] = radar.get("VV", np.nan)
        features["VH"] = radar.get("VH", np.nan)
    except Exception as exc:
        print(f"  ⚠️  Sentinel-1 fetch failed: {exc}")
        features["VV"] = np.nan
        features["VH"] = np.nan

    # ── ERA5-Land climate ─────────────────────────────────────────────────────
    try:
        era5 = (
            ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
            .filterBounds(point)
            .filterDate("2023-01-01", "2024-01-01")
            .select(["temperature_2m", "total_precipitation_sum"])
            .mean()
        )
        climate = era5.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=11132,
            maxPixels=1e6
        ).getInfo()

        temp_k  = climate.get("temperature_2m",       None)
        precip  = climate.get("total_precipitation_sum", None)

        features["climate_temp"]   = round(float(temp_k) - 273.15, 2) if temp_k is not None else np.nan
        features["climate_precip"] = round(float(precip) * 1000.0, 2) if precip is not None else np.nan  # m → mm
    except Exception as exc:
        print(f"  ⚠️  ERA5 climate fetch failed: {exc}")
        features["climate_temp"]   = np.nan
        features["climate_precip"] = np.nan

    # ── SRTM elevation ────────────────────────────────────────────────────────
    try:
        srtm = ee.Image("USGS/SRTMGL1_003").select("elevation")
        elev = srtm.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=30,
            maxPixels=1e6
        ).getInfo()
        features["terrain_elevation"] = elev.get("elevation", np.nan)
    except Exception as exc:
        print(f"  ⚠️  SRTM elevation fetch failed: {exc}")
        features["terrain_elevation"] = np.nan

    # Guard: if all key optical bands are NaN, treat as no-data
    key_bands = ["B4", "B8", "NDVI"]
    all_nan = all(
        (v is None or (isinstance(v, float) and math.isnan(v)))
        for v in [features.get(b) for b in key_bands]
    )
    if all_nan:
        return None

    return features


# ────────────────────────────────────────────────────────────────
# FUNCTION 3 — NaN imputation for model input
# ────────────────────────────────────────────────────────────────

def _impute_missing_features(row: dict, ndvi: float) -> dict:
    """
    Fills missing feature values with biome-aware fallbacks.
    Called only when GEE returns partial data (e.g. SAR gap, cloud gap).
    Uses NDVI to scale vegetation indices where possible.
    """
    safe_ndvi = ndvi if (ndvi is not None and not math.isnan(ndvi)) else 0.3

    fallbacks = {
        # SAR — global forest median values (dB)
        "VV": -12.0,
        "VH": -19.0,
        # Optical bands — moderate tropical broadleaf defaults
        "B2": 500.0,
        "B4": 450.0,
        "B5": 1100.0,
        "B6": 2200.0,
        "B7": 2500.0,
        "B8": 2800.0,
        # Derived indices — scaled from observed NDVI
        "EVI":  round(max(0.0, safe_ndvi * 0.70), 4),
        "NDRE": round(max(0.0, safe_ndvi * 0.60), 4),
        # Climate — tropical/subtropical defaults
        "climate_temp":   22.0,
        "climate_precip": 1400.0,
        "terrain_elevation": 300.0,
    }

    imputed = {}
    for key, val in row.items():
        if val is None or (isinstance(val, float) and math.isnan(val)):
            imputed[key] = fallbacks.get(key, 0.0)
        else:
            imputed[key] = val
    return imputed


# ────────────────────────────────────────────────────────────────
# FUNCTION 4 — Main inference entry point
# ────────────────────────────────────────────────────────────────

FEATURE_COLS = [
    "B2", "B4", "B5", "B6", "B7", "B8",
    "VV", "VH",
    "climate_temp", "climate_precip", "terrain_elevation",
    "NDVI", "EVI", "NDRE",
]

def universal_eco_inference(
    target_lat: float,
    target_lon: float,
    model=None,
    save_model_path: str | None = None,
) -> dict:
    """
    Infers biomass density and carbon offset for any coordinate on Earth.

    Parameters
    ──────────
    target_lat      : float  — latitude  in [-90, 90]
    target_lon      : float  — longitude in [-180, 180]
    model           : sklearn estimator (optional). If None, returns raw GEE
                      features without a numeric prediction.
    save_model_path : str (optional) — path to write model .pkl after inference.
                      Pass only when you explicitly want to update the file.

    Returns
    ───────
    dict with keys:
        biomass       (float | None)  — above-ground biomass in tons/ha
        carbon_offset (float | None)  — carbon stock in tons C/ha  (AGB × 0.47)
        surface_type  (str)           — e.g. "Land", "Ocean/Water", "Permanent Ice/Snow"
        lc_class      (str)           — raw MODIS class label
        features      (dict | None)   — all satellite features fetched from GEE
    """
    SEP = "=" * 64
    print(SEP)
    print("🛰️   ECOAUDIT AI — UNIVERSAL EARTH INFERENCE")
    print(SEP)
    print(f"📍  Coordinates  →  Lat: {target_lat:>10.4f}   |   Lon: {target_lon:>11.4f}")

    # ── Coordinate range validation ──────────────────────────────────────────
    if not (-90.0 <= target_lat <= 90.0) or not (-180.0 <= target_lon <= 180.0):
        print("❌  Invalid coordinates. Lat ∈ [-90, 90], Lon ∈ [-180, 180].")
        print(SEP + "\n")
        return {
            "biomass": None, "carbon_offset": None,
            "surface_type": "Invalid", "lc_class": "Invalid", "features": None,
        }

    # ── Step 1: Surface classification ──────────────────────────────────────
    print("🔍  Querying MODIS land cover (500 m) via GEE …")
    is_vegetated, surface_label = classify_surface(target_lat, target_lon)

    print(f"🗺️   Surface Type : {surface_label}")
    print("-" * 64)

    # ── Non-vegetated surfaces → return 0.0 immediately ─────────────────────
    if not is_vegetated:
        print(f"   🌿 NDVI              : N/A")
        print(f"   🌡️  Temperature (°C)  : N/A")
        print(f"   💧 Precipitation     : N/A")
        print(f"   ⛰️  Elevation (m)     : N/A")
        print(f"   📊 Biomass Density   : 0.00 tons/ha")
        print(f"   💰 Carbon Offset     : 0.00 tons C/ha")
        print(SEP + "\n")
        return {
            "biomass": 0.0,
            "carbon_offset": 0.0,
            "surface_type": surface_label,
            "lc_class": surface_label,
            "features": None,
        }

    # ── Step 2: Fetch live GEE satellite features ────────────────────────────
    print("📡  Fetching live satellite telemetry …")
    raw_features = fetch_gee_features(target_lat, target_lon)

    if raw_features is None:
        print("⚠️   No satellite data returned for this location.")
        print("    Possible causes: polar night, persistent cloud cover, data gap.")
        print("    Try widening the date range in fetch_gee_features() or")
        print("    increasing the cloud filter threshold from 20 → 30 %.")
        print(SEP + "\n")
        return {
            "biomass": None, "carbon_offset": None,
            "surface_type": "Land (No Data)", "lc_class": surface_label,
            "features": None,
        }

    # ── Print fetched telemetry ──────────────────────────────────────────────
    def _fmt(val, decimals=4):
        if val is None or (isinstance(val, float) and math.isnan(val)):
            return "N/A"
        return f"{val:.{decimals}f}"

    ndvi_val = raw_features.get("NDVI")
    print(f"   🌿 NDVI              : {_fmt(ndvi_val)}")
    print(f"   🌡️  Temperature (°C)  : {_fmt(raw_features.get('climate_temp'), 1)}")
    print(f"   💧 Precipitation(mm) : {_fmt(raw_features.get('climate_precip'), 1)}")
    print(f"   ⛰️  Elevation (m)     : {_fmt(raw_features.get('terrain_elevation'), 0)}")
    print(f"   📡 VV backscatter    : {_fmt(raw_features.get('VV'), 2)} dB")
    print(f"   📡 VH backscatter    : {_fmt(raw_features.get('VH'), 2)} dB")

    # ── Step 3: Model inference ──────────────────────────────────────────────
    predicted_agb     = None
    calculated_offset = None

    if model is not None:
        row = {col: raw_features.get(col, np.nan) for col in FEATURE_COLS}

        # Detect missing values and impute
        missing = [k for k, v in row.items()
                   if v is None or (isinstance(v, float) and math.isnan(v))]
        if missing:
            print(f"   ⚠️  Imputing {len(missing)} missing band(s): {missing}")
            row = _impute_missing_features(row, ndvi_val)

        test_df = pd.DataFrame([row])[FEATURE_COLS]

        try:
            predicted_agb    = float(model.predict(test_df)[0])
            predicted_agb    = max(0.0, predicted_agb)   # biomass is non-negative
            calculated_offset = round(predicted_agb * 0.47, 4)  # IPCC: AGB × 0.47 = C stock
            predicted_agb    = round(predicted_agb, 4)

            print(f"   📊 Biomass Density   : {predicted_agb:.2f} tons/ha")
            print(f"   💰 Carbon Offset     : {calculated_offset:.2f} tons C/ha")

        except Exception as exc:
            print(f"❌  Model prediction failed: {exc}")
            predicted_agb     = None
            calculated_offset = None
    else:
        print("   ℹ️   No model provided — raw GEE features returned only.")

    print(SEP + "\n")

    # ── Step 4: Optionally save model (decoupled — only when explicitly asked) ──
    if save_model_path is not None and model is not None:
        try:
            with open(save_model_path, "wb") as fh:
                pickle.dump(model, fh)
            print(f"✅  Model saved → '{save_model_path}'\n")
        except Exception as exc:
            print(f"⚠️   Model save failed: {exc}\n")

    return {
        "biomass":       predicted_agb,
        "carbon_offset": calculated_offset,
        "surface_type":  surface_label,
        "lc_class":      surface_label,
        "features":      raw_features,
    }


# ────────────────────────────────────────────────────────────────
# HELPER — Save / load model
# ────────────────────────────────────────────────────────────────

def save_model(model, path: str = "carbon_model.pkl") -> None:
    """Saves a trained sklearn model to disk. Call once after training — not per inference."""
    with open(path, "wb") as fh:
        pickle.dump(model, fh)
    print(f"✅  Model saved → '{path}'")


def load_model(path: str = "carbon_model.pkl"):
    """Loads a previously saved model from disk."""
    with open(path, "rb") as fh:
        model = pickle.load(fh)
    print(f"✅  Model loaded ← '{path}'")
    return model


print("✅  Inference engine loaded. Proceed to Cell 3 to run predictions.")


# ================================================================
# CELL 3 — MANUAL COORDINATE CHECKER
# ================================================================
# ► Change TARGET_LAT / TARGET_LON to any coordinate on Earth.
# ► Re-run this cell to get a fresh prediction.
# ► Set SAVE_MODEL = True only when you want to update carbon_model.pkl.
# ================================================================

# ── Load (or skip) the trained model ────────────────────────────────────────
try:
    trained_model = load_model("carbon_model.pkl")
except FileNotFoundError:
    print("⚠️  No saved model found — set trained_model = global_raster_model")
    print("   if you trained it earlier in the session, or train it first.\n")
    # If you trained the model in this session, assign it manually:
    # trained_model = global_raster_model
    trained_model = None   # returns raw GEE features without a prediction

# ── Set your target coordinate here ─────────────────────────────────────────
TARGET_LAT =  -3.4653    # ← change me  (range: -90.0 to 90.0)
TARGET_LON = -62.2159    # ← change me  (range: -180.0 to 180.0)

SAVE_MODEL  = False      # set True to write model to carbon_model.pkl after this call

# ── Run inference ────────────────────────────────────────────────────────────
result = universal_eco_inference(
    target_lat=TARGET_LAT,
    target_lon=TARGET_LON,
    model=trained_model,
    save_model_path="carbon_model.pkl" if SAVE_MODEL else None,
)

# ── Inspect raw GEE features ─────────────────────────────────────────────────
if result["features"] is not None:
    feat_df = pd.DataFrame([result["features"]]).T
    feat_df.columns = ["value"]
    feat_df.index.name = "feature"
    print("\n📋  Raw satellite features pulled from GEE:")
    print(feat_df.to_string())
else:
    print("ℹ️   No satellite features returned (ocean, ice, or data gap).")

# ── Final result summary ──────────────────────────────────────────────────────
print("\n" + "=" * 64)
print("  PREDICTION SUMMARY")
print("=" * 64)
print(f"  Coordinate    : ({TARGET_LAT}, {TARGET_LON})")
print(f"  Surface type  : {result['surface_type']}")

if result["biomass"] is not None:
    print(f"  Biomass       : {result['biomass']:.2f} tons / hectare")
    print(f"  Carbon offset : {result['carbon_offset']:.2f} tons C / hectare")
else:
    print("  Biomass       : N/A")
    print("  Carbon offset : N/A")
print("=" * 64)


# ================================================================
# CELL 4 (OPTIONAL) — BATCH TEST MATRIX
# ================================================================
# Runs a set of known reference points and prints a summary table.
# Useful for sanity-checking model predictions across biome types.
# ================================================================

REFERENCE_POINTS = [
    # (lat,       lon,       label,                      expected biomass range tons/ha)
    ( 11.3700,  142.5900,  "Mariana Trench",             "0.0  (ocean)"),
    (-48.8700, -123.3900,  "Point Nemo",                 "0.0  (ocean)"),
    (  0.0000,    0.0000,  "Gulf of Guinea",             "0.0  (ocean)"),
    (-80.1747,   90.6550,  "East Antarctic Ice Sheet",   "0.0  (permanent ice)"),
    ( 71.2000, -156.7000,  "Alaska Arctic tundra",       "0.5–9   (tundra)"),
    ( 78.2000,   15.6000,  "Svalbard tundra",            "0.5–30  (tundra)"),
    ( 23.4162,   25.6628,  "Sahara Desert",              "0–5    (desert)"),
    ( 62.0000,  105.0000,  "Siberian taiga",             "50–200  (boreal)"),
    ( -3.4653,  -62.2159,  "Amazon Basin rainforest",    "200–400 (tropical)"),
    ( -0.7893,   24.0850,  "Congo Basin",                "200–350 (tropical)"),
    ( 36.7783, -119.4179,  "Central California farms",   "5–80    (agri)"),
    ( 51.5074,   -0.1278,  "London (urban)",             "0–15    (urban)"),
    (34.27,  77.85,  "Antarctic Peninsula coast",  "1–10    (polar tundra)"),
]

print("\n" + "=" * 90)
print("  BATCH TEST MATRIX")
print("=" * 90)
print(f"  {'Location':<35} {'Surface':<22} {'Biomass':>10}  {'C-Offset':>10}  {'Expected range':>20}")
print("  " + "─" * 86)

batch_results = {}
for lat, lon, label, expected in REFERENCE_POINTS:
    r = universal_eco_inference(lat, lon, model=trained_model)
    batch_results[label] = r

    bio = f"{r['biomass']:.2f}" if r["biomass"] is not None else "N/A"
    co  = f"{r['carbon_offset']:.2f}" if r["carbon_offset"] is not None else "N/A"
    print(f"  {label:<35} {r['surface_type']:<22} {bio:>10}  {co:>10}  {expected:>20}")

print("=" * 90)

# Save model once after all batch inference (not per-call)
if trained_model is not None and SAVE_MODEL:
    save_model(trained_model, "carbon_model.pkl")

✅  Google Earth Engine initialised.
✅  Inference engine loaded. Proceed to Cell 3 to run predictions.
⚠️  No saved model found — set trained_model = global_raster_model
   if you trained it earlier in the session, or train it first.

🛰️   ECOAUDIT AI — UNIVERSAL EARTH INFERENCE
📍  Coordinates  →  Lat:    -3.4653   |   Lon:    -62.2159
🔍  Querying MODIS land cover (500 m) via GEE …
🗺️   Surface Type : Land
----------------------------------------------------------------
📡  Fetching live satellite telemetry …
   🌿 NDVI              : 0.8163
   🌡️  Temperature (°C)  : 27.1
   💧 Precipitation(mm) : 174.4
   ⛰️  Elevation (m)     : 48
   📡 VV backscatter    : -8.24 dB
   📡 VH backscatter    : -14.41 dB
   ℹ️   No model provided — raw GEE features returned only.


📋  Raw satellite features pulled from GEE:
                         value
feature                       
B2                  330.320444
B4                  306.410413
B5                  845.940043
B6                 2395.084189
B7

In [ ]:
# ==============================================================================
# CELL 2 — CORE INFERENCE ENGINE (Optimized: Allowed Urban & Wetland Biomass)
# ==============================================================================

# Only exclude open water (0, 17) and permanent ice sheets (15).
# class 13 (Urban) and class 11 (Wetlands) are now analyzed for real biomass!
NON_VEGETATED_CLASSES = {0, 15, 17}

SURFACE_LABELS = {
    0:  "Ocean/Water",
    11: "Wetland",
    13: "Urban/Built-up",
    15: "Permanent Ice/Snow",
    17: "Water Body",
}

def classify_surface(lat: float, lon: float) -> tuple[bool, str]:
    point = ee.Geometry.Point([lon, lat])
    try:
        result = (
            ee.ImageCollection("MODIS/006/MCD12Q1")
            .filterDate("2020-01-01", "2021-01-01")
            .first()
            .select("LC_Type1")
            .reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=500,
                maxPixels=1
            )
            .getInfo()
        )
        lc_value = result.get("LC_Type1")

        if lc_value is None:
            return False, "Ocean/Water"

        lc_int = int(lc_value)

        # Only exclude water and permanent ice
        if lc_int in NON_VEGETATED_CLASSES:
            label = SURFACE_LABELS.get(lc_int, f"Non-vegetated (class {lc_int})")
            return False, label

        # Urban (13) and Wetland (11) are treated as Land and allowed to process!
        label = SURFACE_LABELS.get(lc_int, "Land")
        return True, label

    except Exception as exc:
        try:
            from global_land_mask import globe
            on_land = bool(globe.is_land(lat, lon))
            return on_land, ("Land" if on_land else "Ocean/Water")
        except Exception:
            return False, f"Classification error: {exc}"

def fetch_gee_features(lat: float, lon: float) -> dict | None:
    point  = ee.Geometry.Point([lon, lat])
    region = point.buffer(1000)   # 1 km buffer for stable pixel sampling
    features = {}

    # ── Sentinel-2 optical ────────────────────────────────────────────────────
    try:
        s2 = (
            ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(point)
            .filterDate("2023-01-01", "2024-01-01")
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
            .median()
            .select(["B2", "B4", "B5", "B6", "B7", "B8"])
        )
        optical = s2.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=10,
            maxPixels=1e6
        ).getInfo()

        for band in ["B2", "B4", "B5", "B6", "B7", "B8"]:
            features[band] = optical.get(band, np.nan)

        # Vegetation indices
        b2 = features["B2"] if features["B2"] and not (isinstance(features["B2"], float) and math.isnan(features["B2"])) else 0.0
        b4 = features["B4"] if features["B4"] and not (isinstance(features["B4"], float) and math.isnan(features["B4"])) else 1e-6
        b5 = features["B5"] if features["B5"] and not (isinstance(features["B5"], float) and math.isnan(features["B5"])) else 0.0
        b8 = features["B8"] if features["B8"] and not (isinstance(features["B8"], float) and math.isnan(features["B8"])) else 0.0

        eps = 1e-6
        ndvi = (b8 - b4) / (b8 + b4 + eps)
        evi  = 2.5 * (b8 - b4) / (b8 + 6.0 * b4 - 7.5 * b2 + 10000.0 + eps)
        ndre = (b5 - b4) / (b5 + b4 + eps)

        features["NDVI"] = round(float(ndvi), 6)
        features["EVI"]  = round(float(evi),  6)
        features["NDRE"] = round(float(ndre), 6)

    except Exception as exc:
        print(f"  ⚠️  Sentinel-2 fetch failed: {exc}")
        for k in ["B2", "B4", "B5", "B6", "B7", "B8", "NDVI", "EVI", "NDRE"]:
            features[k] = np.nan

    # ── Sentinel-1 SAR ────────────────────────────────────────────────────────
    try:
        s1 = (
            ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(point)
            .filterDate("2023-01-01", "2024-01-01")
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .select(["VV", "VH"])
            .mean()
        )
        radar = s1.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=10,
            maxPixels=1e6
        ).getInfo()
        features["VV"] = radar.get("VV", np.nan)
        features["VH"] = radar.get("VH", np.nan)
    except Exception as exc:
        print(f"  ⚠️  Sentinel-1 fetch failed: {exc}")
        features["VV"] = np.nan
        features["VH"] = np.nan

    # ── ERA5-Land climate ─────────────────────────────────────────────────────
    try:
        era5 = (
            ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
            .filterBounds(point)
            .filterDate("2023-01-01", "2024-01-01")
            .select(["temperature_2m", "total_precipitation_sum"])
            .mean()
        )
        climate = era5.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=11132,
            maxPixels=1e6
        ).getInfo()

        temp_k  = climate.get("temperature_2m",       None)
        precip  = climate.get("total_precipitation_sum", None)

        features["climate_temp"]   = round(float(temp_k) - 273.15, 2) if temp_k is not None else np.nan
        features["climate_precip"] = round(float(precip) * 1000.0, 2) if precip is not None else np.nan  # m → mm
    except Exception as exc:
        print(f"  ⚠️  ERA5 climate fetch failed: {exc}")
        features["climate_temp"]   = np.nan
        features["climate_precip"] = np.nan

    # ── SRTM elevation ────────────────────────────────────────────────────────
    try:
        srtm = ee.Image("USGS/SRTMGL1_003").select("elevation")
        elev = srtm.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=30,
            maxPixels=1e6
        ).getInfo()
        features["terrain_elevation"] = elev.get("elevation", np.nan)
    except Exception as exc:
        print(f"  ⚠️  SRTM elevation fetch failed: {exc}")
        features["terrain_elevation"] = np.nan

    # Guard: if all key optical bands are NaN, treat as no-data
    key_bands = ["B4", "B8", "NDVI"]
    all_nan = all(
        (v is None or (isinstance(v, float) and math.isnan(v)))
        for v in [features.get(b) for b in key_bands]
    )
    if all_nan:
        return None

    return features

def _impute_missing_features(row: dict, ndvi: float) -> dict:
    safe_ndvi = ndvi if (ndvi is not None and not math.isnan(ndvi)) else 0.3
    fallbacks = {
        "VV": -12.0, "VH": -19.0,
        "B2": 500.0, "B4": 450.0, "B5": 1100.0, "B6": 2200.0, "B7": 2500.0, "B8": 2800.0,
        "EVI":  round(max(0.0, safe_ndvi * 0.70), 4),
        "NDRE": round(max(0.0, safe_ndvi * 0.60), 4),
        "climate_temp":   22.0,
        "climate_precip": 1400.0,
        "terrain_elevation": 300.0,
    }
    imputed = {}
    for key, val in row.items():
        if val is None or (isinstance(val, float) and math.isnan(val)):
            imputed[key] = fallbacks.get(key, 0.0)
        else:
            imputed[key] = val
    return imputed

FEATURE_COLS = [
    "B2", "B4", "B5", "B6", "B7", "B8",
    "VV", "VH",
    "climate_temp", "climate_precip", "terrain_elevation",
    "NDVI", "EVI", "NDRE",
]

def universal_eco_inference(
    target_lat: float,
    target_lon: float,
    model=None,
    save_model_path: str | None = None,
) -> dict:
    SEP = "=" * 64
    print(SEP)
    print("🛰️   ECOAUDIT AI — UNIVERSAL EARTH INFERENCE")
    print(SEP)
    print(f"📍  Coordinates  →  Lat: {target_lat:>10.4f}   |   Lon: {target_lon:>11.4f}")

    if not (-90.0 <= target_lat <= 90.0) or not (-180.0 <= target_lon <= 180.0):
        print("❌  Invalid coordinates. Lat ∈ [-90, 90], Lon ∈ [-180, 180].")
        print(SEP + "\n")
        return {
            "biomass": None, "carbon_offset": None,
            "surface_type": "Invalid", "lc_class": "Invalid", "features": None,
        }

    # classify surface (Urban / Wetland are allowed to pass through now)
    is_vegetated, surface_label = classify_surface(target_lat, target_lon)

    print(f"🗺️   Surface Type : {surface_label}")
    print("-" * 64)

    if not is_vegetated:
        print(f"   🌿 NDVI              : N/A")
        print(f"   🌡️  Temperature (°C)  : N/A")
        print(f"   💧 Precipitation     : N/A")
        print(f"   ⛰️  Elevation (m)     : N/A")
        print(f"   📊 Biomass Density   : 0.00 tons/ha")
        print(f"   💰 Carbon Offset     : 0.00 tons C/ha")
        print(SEP + "\n")
        return {
            "biomass": 0.0, "carbon_offset": 0.0,
            "surface_type": surface_label, "lc_class": surface_label, "features": None,
        }

    print("📡  Fetching live satellite telemetry …")
    raw_features = fetch_gee_features(target_lat, target_lon)

    if raw_features is None:
        print("⚠️   No satellite data returned for this location.")
        print(SEP + "\n")
        return {
            "biomass": None, "carbon_offset": None,
            "surface_type": f"{surface_label} (No Data)", "lc_class": surface_label,
            "features": None,
        }

    def _fmt(val, decimals=4):
        if val is None or (isinstance(val, float) and math.isnan(val)):
            return "N/A"
        return f"{val:.{decimals}f}"

    ndvi_val = raw_features.get("NDVI")
    print(f"   🌿 NDVI              : {_fmt(ndvi_val)}")
    print(f"   🌡️  Temperature (°C)  : {_fmt(raw_features.get('climate_temp'), 1)}")
    print(f"   💧 Precipitation(mm) : {_fmt(raw_features.get('climate_precip'), 1)}")
    print(f"   ⛰️  Elevation (m)     : {_fmt(raw_features.get('terrain_elevation'), 0)}")
    print(f"   📡 VV backscatter    : {_fmt(raw_features.get('VV'), 2)} dB")
    print(f"   📡 VH backscatter    : {_fmt(raw_features.get('VH'), 2)} dB")

    predicted_agb     = None
    calculated_offset = None

    if model is not None:
        row = {col: raw_features.get(col, np.nan) for col in FEATURE_COLS}
        missing = [k for k, v in row.items() if v is None or (isinstance(v, float) and math.isnan(v))]
        if missing:
            print(f"   ⚠️  Imputing {len(missing)} missing band(s): {missing}")
            row = _impute_missing_features(row, ndvi_val)

        test_df = pd.DataFrame([row])[FEATURE_COLS]

        try:
            predicted_agb    = float(model.predict(test_df)[0])
            predicted_agb    = max(0.0, predicted_agb)
            calculated_offset = round(predicted_agb * 0.47, 4)
            predicted_agb    = round(predicted_agb, 4)

            print(f"   📊 Biomass Density   : {predicted_agb:.2f} tons/ha")
            print(f"   💰 Carbon Offset     : {calculated_offset:.2f} tons C/ha")

        except Exception as exc:
            print(f"❌  Model prediction failed: {exc}")
            predicted_agb     = None
            calculated_offset = None
    else:
        print("   ℹ️   No model provided — raw GEE features returned only.")

    print(SEP + "\n")

    if save_model_path is not None and model is not None:
        try:
            with open(save_model_path, "wb") as fh:
                pickle.dump(model, fh)
            print(f"✅  Model saved → '{save_model_path}'\n")
        except Exception as exc:
            print(f"⚠️   Model save failed: {exc}\n")

    return {
        "biomass":       predicted_agb,
        "carbon_offset": calculated_offset,
        "surface_type":  surface_label,
        "lc_class":      surface_label,
        "features":      raw_features,
    }

In [ ]:
# ==============================================================================
# CELL 3 — COORDINATE CHECKER (Altered for your GEE-trained model)
# ==============================================================================

# 1. Load the model from the saved file
try:
    model_data = load_model("carbon_model.pkl")

    # Check if the loaded model is our dictionary wrapper
    if isinstance(model_data, dict):
        global_raster_model = model_data["model"]

        # DYNAMIC FIX: Update the global FEATURE_COLS variable so the checker
        # uses the exact 12 bands the model was trained on
        FEATURE_COLS = model_data["feature_cols"]
        print(f"✅ Loaded GEE-trained model expecting {len(FEATURE_COLS)} features.")
    else:
        global_raster_model = model_data
        print("✅ Loaded standard model.")
except FileNotFoundError:
    print("⚠️  No saved model found — running in feature-only mode.\n")
    global_raster_model = None

# ── Set your target coordinate here ─────────────────────────────────────────
TARGET_LAT = -75    # ← Feel free to change
TARGET_LON = 0   # ← Feel free to change

# ── Run the inference engine ────────────────────────────────────────────────
result = universal_eco_inference(
    target_lat=TARGET_LAT,
    target_lon=TARGET_LON,
    model=global_raster_model,
)

# ── Inspect raw GEE features ─────────────────────────────────────────────────
if result["features"]:
    import pandas as pd
    feat_df = pd.DataFrame([result["features"]])
    print("\n📋 Raw satellite features:")
    print(feat_df.T.to_string(header=False))
else:
    print("No feature data returned (ocean or data gap).")

# ── Pretty result summary ────────────────────────────────────────────────────
print("\n" + "="*50)
print(f"  Coordinate    : ({TARGET_LAT}, {TARGET_LON})")
print(f"  Surface type  : {result['surface_type']}")
if result["biomass"] is not None:
    print(f"  Biomass       : {result['biomass']:.2f} tons/ha")
    print(f"  Carbon offset : {result['carbon_offset']:.2f} tons C/ha")
else:
    print("  Biomass       : N/A")
    print("  Carbon offset : N/A")
print("="*50)

✅  Model loaded ← 'carbon_model.pkl'
✅ Loaded GEE-trained model expecting 12 features.
🛰️   ECOAUDIT AI — UNIVERSAL EARTH INFERENCE
📍  Coordinates  →  Lat:   -75.0000   |   Lon:      0.0000
🗺️   Surface Type : Permanent Ice/Snow
----------------------------------------------------------------
   🌿 NDVI              : N/A
   🌡️  Temperature (°C)  : N/A
   💧 Precipitation     : N/A
   ⛰️  Elevation (m)     : N/A
   📊 Biomass Density   : 0.00 tons/ha
   💰 Carbon Offset     : 0.00 tons C/ha

No feature data returned (ocean or data gap).

  Coordinate    : (-75, 0)
  Surface type  : Permanent Ice/Snow
  Biomass       : 0.00 tons/ha
  Carbon offset : 0.00 tons C/ha
